In [13]:
from __future__ import print_function
from builtins import str
from builtins import range
import json

%matplotlib inline
from os.path import join as opj
import json
from nipype.interfaces.spm import Level1Design, EstimateModel, EstimateContrast
from nipype.algorithms.modelgen import SpecifySPMModel
from nipype.interfaces.utility import Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
from nipype import Workflow, Node
from nipype.interfaces.nipy.model import FitGLM, EstimateContrast
from nipype.interfaces.nipy.preprocess import ComputeMask

In [14]:
import nipype.interfaces.io as nio  # Data i/o
import nipype.interfaces.spm as spm  # spm
import nipype.interfaces.matlab as mlab  # how to run matlab
import nipype.interfaces.utility as util  # utility
import nipype.pipeline.engine as pe  # pypeline engine
import nipype.algorithms.rapidart as ra  # artifact detection
import nipype.algorithms.modelgen as model  # model specification
from nipype import Workflow, Node
import os  # system functions

# sTART with smoothing 

In [29]:
from nipype.interfaces.spm import Smooth
# Smoothing widths to apply
fwhm = [4, 8]

# Smooth - image smoothing
smooth = Node(Smooth(), name="smooth")
smooth.iterables = ("fwhm", fwhm)




NameError: name 'preproc' is not defined

In [15]:
# Set the way matlab should be called
mlab.MatlabCommand.set_default_matlab_cmd("matlab -nodesktop -nosplash")

In [16]:
# Specify the location of the data.
data_dir = os.path.abspath('/Users/sbedi/data/ds-stressrisk')
experiment_dir = os.path.join(data_dir, "derivatives", "fmriprep")
output_dir = os.path.join(data_dir, "derivatives", "glm_classic")
working_dir = os.path.join(data_dir, "derivatives", "nipype_glm_workingdir")

# Specify the subject directories
subject_list = ['sub-01']
# Map field names to individual subject runs.
info = dict(
    func=[['subject_id', ['f3', 'f5', 'f7', 'f10']]],
    struct=[['subject_id', 'struct']])

infosource = pe.Node(
    interface=util.IdentityInterface(fields=['subject_id']), name="infosource")

# TR of functional images
fson_f = os.path.join(data_dir, 'task-risk_bold.json')
with open(fson_f, 'rt') as fp: #fson_f
    task_info = json.load(fp)
TR = task_info['RepetitionTime']

# Smoothing withds used during preprocessing
fwhm = [4, 8]

In [17]:
TR

2.298

In [18]:
# SpecifyModel - Generates SPM-specific Model
modelspec = Node(SpecifySPMModel(concatenate_runs=False,
                                 input_units='secs',
                                 output_units='secs',
                                 time_repetition=TR,
                                 high_pass_filter_cutoff=128),
                 name="modelspec")

# Level1Design - Generates an SPM design matrix
level1design = Node(Level1Design(bases={'hrf': {'derivs': [1, 0]}},
                                 timing_units='secs',
                                 interscan_interval=TR,
                                 model_serial_correlations='FAST'),
                    name="level1design")

# EstimateModel - estimate the parameters of the model
level1estimate = Node(EstimateModel(estimation_method={'Classical': 1}),
                      name="level1estimate")

# EstimateContrast - estimates contrasts
level1conest = Node(EstimateContrast(), name="level1conest")


230320-14:49:54,609 nipype.interface WARNING:
	 Unable to import ['nipy']; EstimateContrast interface may fail to run


# Specify contrasts

In [19]:
# Condition names
condition_names = ['risky', 'safe']
condition_names = ['risky_first',  'risky_second', 'safe_first','safe_second']
# Contrasts
cont01 = ['average',        'T', condition_names, [1/4., 1/4., 1/4., 1/4.]]
cont02 = ['risky_first',         'T', condition_names, [1, 0, 0, 0]]
cont03 = ['risky_second',           'T', condition_names, [0, 1, 0, 0]]
cont04 = ['safe_first',           'T', condition_names, [0, 0, 1, 0]]
cont05 = ['safe_second',           'T', condition_names, [0, 0, 0, 1]]
# cont05 = ['Finger > others','T', condition_names, [1, -0.5, -0.5]]
# cont08 = ['activation',     'F', [cont02, cont03, cont04]]
# cont09 = ['differences',    'F', [cont05, cont06, cont07]]
contrast_list = [cont01, cont02, cont03, cont04, cont05]


# GLM model

In [20]:
#reading the tsv file and what events happened
!cat /Users/sbedi/data/ds-stressrisk/sub-02/ses-1/func/sub-02_ses-1_task-risk_run-1_events.tsv

trial_nr	onset	trial_type	prob1	prob2	n1	n2	choice
1	13.26243919999979	stimulus 1	0.55	1.0	18.0	10.0	
1	21.46934949999924	stimulus 2	0.55	1.0	18.0	10.0	
1	22.712538699997836	choice					2.0
2	27.90866389999792	stimulus 1	0.55	1.0	23.0	10.0	
2	37.11709789999803	stimulus 2	0.55	1.0	23.0	10.0	
2	38.94092219999948	choice					1.0
3	43.57313480000083	stimulus 1	0.55	1.0	41.0	20.0	
3	49.76218829999925	stimulus 2	0.55	1.0	41.0	20.0	
3	51.001213100000314	choice					2.0
4	55.717712499998015	stimulus 1	0.55	1.0	53.0	20.0	
4	61.923486600000615	stimulus 2	0.55	1.0	53.0	20.0	
4	63.27928030000112	choice					1.0
5	67.86228019999908	stimulus 1	0.55	1.0	21.0	7.0	
5	77.07084329999998	stimulus 2	0.55	1.0	21.0	7.0	
5	78.37669929999902	choice					1.0
6	84.5110265000003	stimulus 1	0.55	1.0	26.0	10.0	
6	93.71958570000061	stimulus 2	0.55	1.0	26.0	10.0	
6	94.89187649999803	choice					1.0
7	101.1598129999984	stimulus 1	0.55	1.0	25.0	14.0	
7	109.36741489999986	stimulus 2	0.55	1.0	25.0	14.0	
7	110.62309879999884	ch

In [21]:
import pandas as pd
trialinfo = pd.read_table('/Users/sbedi/data/ds-stressrisk/sub-02/ses-1/func/sub-02_ses-1_task-risk_run-1_events.tsv')
trialinfo

,trial_nr,onset,trial_type,prob1,prob2,n1,n2,choice
0,1,13.262439,stimulus 1,0.55,1.00,18.0,10.0,NaN
1,1,21.469349,stimulus 2,0.55,1.00,18.0,10.0,NaN
2,1,22.712539,choice,NaN,NaN,NaN,NaN,2.0
3,2,27.908664,stimulus 1,0.55,1.00,23.0,10.0,NaN
4,2,37.117098,stimulus 2,0.55,1.00,23.0,10.0,NaN
5,2,38.940922,choice,NaN,NaN,NaN,NaN,1.0
6,3,43.573135,stimulus 1,0.55,1.00,41.0,20.0,NaN
7,3,49.762188,stimulus 2,0.55,1.00,41.0,20.0,NaN
8,3,51.001213,choice,NaN,NaN,NaN,NaN,2.0
9,4,55.717712,stimulus 1,0.55,1.00,53.0,20.0,NaN


Here we set up iteration over all the subjects. The following line is a particular example of the flexibility of the system. The datasource attribute iterables tells the pipeline engine that it should repeat the analysis on each of the items in the subject_list. In the current example, the entire first level preprocessing and estimation will be repeated for each subject contained in subject_list.

In [22]:
infosource.iterables = ('subject_id', subject_list)


In [23]:
datasource = pe.Node(
    interface=nio.DataGrabber(
        infields=['subject_id'], outfields=['func', 'struct']),
    name='datasource')
datasource.inputs.base_directory = data_dir
datasource.inputs.template = '%s/%s.nii'
datasource.inputs.template_args = info
datasource.inputs.sort_filelist = True

# Set up analysis components
Here we create a function that returns subject-specific information about the experimental paradigm. This is used by the nipype.interfaces.spm.SpecifyModel to create the information necessary to generate an SPM design matrix

In [24]:
def subjectinfo(subject_id):
    from nipype.interfaces.base import Bunch
    from copy import deepcopy
    print("Subject ID: %s\n" % str(subject_id))
    output = []
    names = ['Task-Odd', 'Task-Even']
    for r in range(4):
        onsets = [list(range(15, 240, 60)), list(range(45, 240, 60))]
        output.insert(r,
                      Bunch(
                          conditions=names,
                          onsets=deepcopy(onsets),
                          durations=[[15] for s in names],
                          amplitudes=None,
                          tmod=None,
                          pmod=None,
                          regressor_names=None,
                          regressors=None))
    return output